# 2D GPU Accelerator — Demo Notebook

**Board:** PYNQ-Z2  
**Overlay:** `gpu_accel.bit`  
**Resolution:** 1280 × 720 @ RGB565

This notebook demonstrates hardware-accelerated 2D drawing via the GPU IP core:
- Single pixel plot
- Line drawing (Bresenham, all octants)
- Rectangles, triangles, circles
- Performance benchmark (lines/second)
- Double-buffered display via AXI VDMA + HDMI TX

---

## 1. Load Overlay

In [ ]:
import sys
import time
import numpy as np
from pynq import Overlay

# Add GPU driver to path
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/gpu')
from gpu_draw import GPU, VideoOutput, rgb565
from gpu_draw import RED, GREEN, BLUE, WHITE, BLACK, CYAN, MAGENTA, YELLOW
from gpu_draw import WIDTH, HEIGHT

OVERLAY_PATH = '/home/xilinx/jupyter_notebooks/gpu/gpu_accel.bit'

print('Loading overlay...')
ol = Overlay(OVERLAY_PATH)
print('Overlay loaded:', list(ol.ip_dict.keys()))

## 2. Set Up Video Output & GPU

In [ ]:
print('Setting up video output (double-buffered VDMA)...')
video = VideoOutput(ol.axi_vdma_0, width=WIDTH, height=HEIGHT)

print('Creating GPU driver...')
gpu = GPU(
    ip=ol.gpu_ctrl_axi_0,
    fb_phys_addr=video.back_phys,
    fb_array=video.back_buffer,
    width=WIDTH,
    height=HEIGHT
)

print(f'Framebuffer physical address: 0x{video.back_phys:08X}')
print(f'Resolution: {WIDTH}x{HEIGHT} @ RGB565')
print('GPU status:', gpu.status)

## 3. Clear Screen

In [ ]:
# Software clear (numpy — fast)
gpu.clear(rgb565(0, 0, 40))   # dark navy background
video.flip()
print('Screen cleared.')

## 4. Hardware Line Drawing

In [ ]:
# Diagonal X across the screen
gpu.clear(rgb565(0, 0, 40))

print('Drawing red X...')
gpu.draw_line(0, 0, WIDTH-1, HEIGHT-1, RED)
gpu.draw_line(WIDTH-1, 0, 0, HEIGHT-1, RED)

print('Drawing green border...')
gpu.draw_rect(20, 20, WIDTH-40, HEIGHT-40, GREEN)

print('Drawing white crosshair at center...')
cx, cy = WIDTH // 2, HEIGHT // 2
gpu.draw_hline(cx - 60, cy, 120, WHITE)
gpu.draw_vline(cx, cy - 60, 120, WHITE)

video.flip()
print('Done.')

## 5. Shapes Demo

In [ ]:
gpu.clear(rgb565(10, 10, 30))

# Cyan circle
print('Drawing cyan circle...')
gpu.draw_circle(cx, cy, 250, CYAN, segments=120)

# Yellow triangle
print('Drawing yellow triangle...')
gpu.draw_triangle(cx, 80, 120, HEIGHT-80, WIDTH-120, HEIGHT-80, YELLOW)

# Magenta rectangle filled
print('Drawing filled rectangle (software, numpy)...')
gpu.fill_rect(50, 50, 200, 100, MAGENTA)

# Blue rectangle outline
gpu.draw_rect(50, 50, 200, 100, WHITE)

video.flip()
print('Shapes drawn.')

## 6. Octant Test — All 8 Line Directions

In [ ]:
gpu.clear(BLACK)

cx, cy = WIDTH // 2, HEIGHT // 2
r = 300

import math
colors = [RED, GREEN, BLUE, CYAN, MAGENTA, YELLOW, WHITE, rgb565(255,128,0)]
labels = ['0°','45°','90°','135°','180°','225°','270°','315°']

for i, angle_deg in enumerate(range(0, 360, 45)):
    angle = math.radians(angle_deg)
    x1 = cx + int(r * math.cos(angle))
    y1 = cy + int(r * math.sin(angle))
    gpu.draw_line(cx, cy, x1, y1, colors[i])
    print(f'  Line {labels[i]}: ({cx},{cy}) → ({x1},{y1})')

video.flip()
print('All 8 octants drawn.')

## 7. Performance Benchmark

In [ ]:
import random

NUM_LINES  = 200
NUM_PIXELS = 1000

random.seed(42)

# --- Line benchmark ---
gpu.clear(BLACK)
t0 = time.perf_counter()
for _ in range(NUM_LINES):
    x0 = random.randint(0, WIDTH-1)
    y0 = random.randint(0, HEIGHT-1)
    x1 = random.randint(0, WIDTH-1)
    y1 = random.randint(0, HEIGHT-1)
    c  = random.randint(0, 0xFFFF)
    gpu.draw_line(x0, y0, x1, y1, c)
t1 = time.perf_counter()

elapsed_line = t1 - t0
lines_per_sec = NUM_LINES / elapsed_line
print(f'Line benchmark  : {NUM_LINES} lines in {elapsed_line*1000:.1f} ms')
print(f'                → {lines_per_sec:.0f} lines/second')

# --- Pixel benchmark ---
t0 = time.perf_counter()
for _ in range(NUM_PIXELS):
    x = random.randint(0, WIDTH-1)
    y = random.randint(0, HEIGHT-1)
    c = random.randint(0, 0xFFFF)
    gpu.plot_pixel(x, y, c)
t1 = time.perf_counter()

elapsed_pix = t1 - t0
pixels_per_sec = NUM_PIXELS / elapsed_pix
print(f'Pixel benchmark : {NUM_PIXELS} pixels in {elapsed_pix*1000:.1f} ms')
print(f'                → {pixels_per_sec:.0f} pixels/second')

video.flip()

## 8. Full Demo Scene

In [ ]:
gpu.clear(rgb565(5, 5, 20))    # deep space background

# Outer frame
gpu.draw_rect(10, 10, WIDTH-20, HEIGHT-20, rgb565(80, 80, 80))

# Title bar (filled)
gpu.fill_rect(10, 10, WIDTH-20, 50, rgb565(30, 30, 80))
gpu.draw_hline(10, 60, WIDTH-20, CYAN)

# Star field (random pixels)
rng = np.random.default_rng(0)
stars_x = rng.integers(20, WIDTH-20,  size=300)
stars_y = rng.integers(80, HEIGHT-20, size=300)
for sx, sy in zip(stars_x, stars_y):
    brightness = rng.integers(120, 255)
    gpu.plot_pixel(int(sx), int(sy), rgb565(brightness, brightness, brightness))

# Planet (circle)
gpu.draw_circle(cx, cy, 180, rgb565(0, 180, 255), segments=180)
gpu.draw_circle(cx, cy, 160, rgb565(0, 120, 200), segments=120)

# Orbit ring (ellipse approximated with lines)
for i in range(180):
    a = math.radians(i * 2)
    a2 = math.radians((i+1) * 2)
    rx, ry = 350, 100
    x1_ = cx + int(rx * math.cos(a))
    y1_ = cy + int(ry * math.sin(a))
    x2_ = cx + int(rx * math.cos(a2))
    y2_ = cy + int(ry * math.sin(a2))
    gpu.draw_line(x1_, y1_, x2_, y2_, rgb565(200, 160, 80))

# Satellite
sat_x = cx + 350
gpu.draw_rect(sat_x - 12, cy - 8, 24, 16, WHITE)
gpu.draw_hline(sat_x - 40, cy,  28, rgb565(150, 150, 150))  # solar panel L
gpu.draw_hline(sat_x + 12, cy,  28, rgb565(150, 150, 150))  # solar panel R

video.flip()
print('Full demo scene rendered!')

## 9. Cleanup

In [ ]:
gpu.clear(BLACK)
video.flip()

# Free PYNQ allocations
for fb in video.fb:
    fb.freebuffer()

print('Cleanup complete.')